In [ ]:
import pandas as pd

In [ ]:
import json

def _serialize_for_parquet(val):
	# Keep None/NaN as None, serialize lists/dicts to JSON strings,
	# decode bytes, otherwise cast to str.
	if pd.isna(val):
		return None
	if isinstance(val, (list, dict)):
		try:
			return json.dumps(val, ensure_ascii=False)
		except Exception:
			return str(val)
	if isinstance(val, (bytes, bytearray)):
		try:
			return val.decode('utf-8', errors='replace')
		except Exception:
			return str(val)
	return str(val)

# # Serialize problematic object columns to safe string/None values
# if 'uploaded_pics' in reviews_matched.columns:
# 	reviews_matched['uploaded_pics'] = reviews_matched['uploaded_pics'].apply(_serialize_for_parquet)

# # If you have other object columns that may contain lists/bytes, handle them similarly:
# for col in ['uploaded_pics_links', 'uploaded_pics']:  # add other columns if needed
# 	if col in reviews_matched.columns:
# 		reviews_matched[col] = reviews_matched[col].apply(_serialize_for_parquet)

##### the  matcehd and relevant place_ids

In [ ]:
sp_pids = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/swimmingpool_list_to_use_25112025.csv")
sp_pids = sp_pids.drop_duplicates(subset=['place_id']).reset_index(drop=True)
#drop rows where the primamry place type is swimming sshool and there is no pool id
sp_pids = sp_pids[~((sp_pids['primary_place_type'].str.lower() == 'swimming school') & (sp_pids['pool_id'].isna()))]
sp_pids.to_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/swimmingpool_list_to_use_25112025.csv", index=False)
sp_pids.info()

In [ ]:
sp_pids.place_id.nunique()

In [ ]:
sp_pids.primary_place_type.value_counts()

In [ ]:
sp_pids[((sp_pids['primary_place_type'].str.lower() == 'swimming school') & (sp_pids['pool_id'].isna()))][['place_id', 'primary_place_type', 'place_name']]

In [ ]:
#check total reviews needed
sp_pids.total_reviews.sum()

### which reviews have been collected

In [ ]:
reviews_collected = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/reviews_matched_16112025.parquet")
reviews_collected = reviews_collected.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
reviews_collected.info()

In [ ]:
rev_1 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/clean_datasets/merged_reviews1.parquet")
rev_1 = rev_1.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
rev_1.info()    

In [ ]:
#reviews collected from google places
reviews_set_1 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_reviews.parquet")
reviews_set_1 = reviews_set_1.drop_duplicates(subset=['review_id']).reset_index(drop=True)
reviews_set_1.info()

In [ ]:
reviews_set_2 = pd.read_csv("/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/all_swim_reviews_germany.csv")
reviews_set_2 = reviews_set_2.drop_duplicates(subset=['review_id']).reset_index(drop=True)
reviews_set_2.info()

In [ ]:
missed_reviews_21 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/clean_datasets/merged_reviews1.parquet")
missed_reviews_21 = missed_reviews_21.drop_duplicates(subset=['review_id']).reset_index(drop=True)
missed_reviews_21.info()

In [ ]:
missed_reviews_16 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/merged_reviews.parquet")
missed_reviews_16 = missed_reviews_16.drop_duplicates(subset=['review_id']).reset_index(drop=True)
missed_reviews_16.info()

### merged all reviews

In [ ]:
all_reviews = pd.concat([reviews_collected, rev_1, reviews_set_1,reviews_set_2,missed_reviews_16, missed_reviews_21], ignore_index=True)
all_reviews = all_reviews[all_reviews['place_id'].isin(sp_pids['place_id'])].reset_index(drop=True)
all_reviews = all_reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
all_reviews.info()

In [ ]:
# Serialize object-like columns so pyarrow can write them to parquet
# _serialize_for_parquet is defined in an earlier cell
obj_cols = [c for c in all_reviews.columns if all_reviews[c].dtype == 'object']
for col in obj_cols:
	all_reviews[col] = all_reviews[col].apply(_serialize_for_parquet)

all_reviews.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/collected_reviews_25112025.parquet", index=False)

### still missing and incomplete

In [ ]:
all_reviews = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/collected_reviews_25112025.parquet")

In [ ]:
still_missing = sp_pids[~sp_pids['place_id'].isin(all_reviews['place_id'])][['place_id', 'unique_identifier','place_name','primary_place_type', 'place_subtype','total_reviews', 'google_url']]
still_missing = still_missing.dropna(subset =['total_reviews'])
still_missing.info()

In [ ]:
still_missing.total_reviews.sum()

In [ ]:
#pids incomplete
import numpy as np
reviews_incomp = all_reviews.groupby('place_id').size().reset_index(name='num_reviews_collected')
reviews_incomp = reviews_incomp.merge(sp_pids[['place_id', 'unique_identifier','total_reviews']], on='place_id', how='left')
reviews_incomp['reviews_missing'] = reviews_incomp['total_reviews'] - reviews_incomp['num_reviews_collected']
#reviews_incomp = reviews_incomp[reviews_incomp['reviews_missing'] > 1].reset_index(drop=True)
##percerntage of reviews missing
reviews_incomp['percent_missing'] = np.round((reviews_incomp['reviews_missing'] / reviews_incomp['total_reviews']) * 100, 2)
#those that have more than 10% reviews missing
reviews_incomp= reviews_incomp[reviews_incomp['percent_missing'] > 10]
reviews_incomp

In [ ]:
reviews_incomp.total_reviews.sum()

In [ ]:
reviews_incomp.reviews_missing.sum()

In [ ]:
rerun_reviews = pd.concat([still_missing[['place_id', 'unique_identifier','total_reviews']], reviews_incomp[['place_id', 'unique_identifier','total_reviews']]], axis=0).drop_duplicates().reset_index(drop=True)
# use division (/) not integer floor-division (//) so ceil yields the correct number of chunks
rerun_reviews['expected_jsons'] = np.ceil(rerun_reviews['total_reviews'] / 10).astype(int)

rerun_reviews.info()

In [ ]:
rerun_reviews.total_reviews.sum()

In [ ]:
((int(512653.0/10)/100)*120)

In [ ]:
rerun_reviews.to_csv("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/rerun_missed_reviews.csv", index=False)

In [ ]:
# Safely compute the sum of 'total_reviews' for rows with a translated review.
# If 'total_reviews' is not present in all_reviews, return the count of translated reviews instead.
mask = all_reviews['trans_review'].notna()
if 'total_reviews' in all_reviews.columns:
	result = all_reviews.loc[mask, 'total_reviews'].sum()
else:
	# fallback: number of reviews that have a translation
	result = int(mask.sum())
result

## Adding the rerun reviews 27.11.2025

In [ ]:
collected_reruns1 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251126112335_reviews.parquet")
collected_reruns2 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251126013648_reviews.parquet")
colllected_reruns3 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251125233613_reviews.parquet")
all_rerun_reviews = pd.concat([collected_reruns1, collected_reruns2, colllected_reruns3], ignore_index=True)
all_rerun_reviews = all_rerun_reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
all_rerun_reviews = all_rerun_reviews[~all_rerun_reviews['review_id'].isin(all_reviews['review_id'])].reset_index(drop=True)
all_rerun_reviews.info()

In [ ]:
all_rerun_reviews.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/rerun_missed_reviews_25112025.parquet")

In [ ]:
all_available_reviews = pd.concat([all_reviews, all_rerun_reviews], ignore_index=True)
all_available_reviews = all_available_reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
all_available_reviews.info()

### save all reviews, including reruns

In [ ]:
all_available_reviews.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/collected_reviews_25112025.parquet", index=False)

In [ ]:
import numpy as np
reviews_incomp = all_available_reviews.groupby('place_id').size().reset_index(name='num_reviews_collected')
reviews_incomp = reviews_incomp.merge(sp_pids[['place_id','place_name', 'unique_identifier','total_reviews']], on='place_id', how='left')
reviews_incomp['reviews_missing'] = reviews_incomp['total_reviews'] - reviews_incomp['num_reviews_collected']
#reviews_incomp = reviews_incomp[reviews_incomp['reviews_missing'] > 1].reset_index(drop=True)
##percerntage of reviews missing
reviews_incomp['percent_missing'] = np.round((reviews_incomp['reviews_missing'] / reviews_incomp['total_reviews']) * 100, 2)
#those that have more than 10% reviews missing
reviews_incomp= reviews_incomp[reviews_incomp['percent_missing'] >0].sort_values('percent_missing', ascending=False).reset_index(drop=True)
reviews_incomp

In [ ]:
reviews_incomp.reviews_missing.sum()

In [ ]:
sp_pids.total_reviews.sum()

In [ ]:
reviews_incomp.reviews_missing.sum()/sp_pids.total_reviews.sum()

In [ ]:
details_reruns1 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251126112335_details.parquet")
details_reruns2 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251126013648_details.parquet")
details_reruns3 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_results20251125233613_details.parquet")
all_rerun_details = pd.concat([details_reruns1, details_reruns2, details_reruns3], ignore_index=True)
#all_rerun_details = all_rerun_details.drop_duplicates(subset=["place_id"]).reset_index(drop=True)
all_rerun_details.info()

In [ ]:
all_rerun_details[['retrieved_time']]

In [ ]:
# convert the retrieved time to time stamp
# then for each place id, get row with the latest retrieved time
all_rerun_details['retrieved_time'] = pd.to_datetime(all_rerun_details['retrieved_time'], errors='coerce')
latest_details = all_rerun_details.sort_values('retrieved_time').groupby('place_id').tail(1).reset_index(drop=True)
latest_details.info()

In [ ]:
latest_details[latest_details['place_id'].isin(reviews_incomp['place_id'])]

In [ ]:
nxt_page  = pd.read_csv("/home/ethel/Desktop/WEBCONF_25/missing_reviews_25112025/mr_nxtpg.csv")
nxt_page.info()

In [ ]:
import pandas as pd
reviews = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/all_collected_reviews_25112025.parquet")
reviews = reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
reviews = reviews.dropna(subset=['trans_review']).reset_index(drop=True)
print(reviews.shape)
reviews.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/all_collected_text_reviews_25112025.parquet", index=False)

In [ ]:
chk0 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/text_reviews_summarized_25112025_checkpoints_992ed0a2/chunk_0.parquet")
chk0.info()

In [ ]:
chk0[['cleaned_text', 'summary']].head()

In [ ]:
import pandas as pd
all_reviews = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/all_collected_reviews_25112025.parquet")
all_reviews[['reviewer_name']]


In [ ]:
name_eth = all_reviews[['reviewer_name']].drop_duplicates().reset_index(drop=True)

In [ ]:
name_eth.to_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/reviewer_names_01122025.csv", index=False)